# 13 — Merge All Job Sources

Merges all 14 per-source standardized CSVs into `final_jobs_dataset.csv`.

**Input:** `data/processed/jobs/*_standardized.csv` (14 files)  
**Output:** `data/processed/jobs/final_jobs_dataset.csv`

> Only re-run if any individual source CSV has been updated.

In [1]:
from pathlib import Path
import os, re
import pandas as pd

BASE_DIR = Path.cwd().parent.parent
JOBS_DIR = BASE_DIR / "data" / "processed" / "jobs"

## Schema and constants

In [2]:
AGGREGATORS = {"linkedin", "staff.am", "job.am"}

OUT_COLS = [
    "source", "source_type", "source_url", "job_title", "company_name", "location",
    "employment_type", "seniority_level", "industries",
    "posting_date", "deadline", "skills_tags", "full_text",
]

## Helper functions

In [3]:
def normalize_date(val):
    """Normalize various date formats to YYYY-MM-DD; blank on failure."""
    if not val or (isinstance(val, float)):
        return ""
    s = str(val).strip()
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", s)
    if m:
        return f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    return s


def seniority_from_title(title: str) -> str:
    """Derive a seniority label from the job title when not provided."""
    t = str(title).lower()
    if any(k in t for k in ["intern", "trainee", "student"]):
        return "Internship"
    if any(k in t for k in ["principal", "staff ", "distinguished", "fellow"]):
        return "Principal"
    if any(k in t for k in ["senior", "sr.", "sr "]):
        return "Senior"
    if any(k in t for k in ["junior", "jr.", "jr "]):
        return "Junior"
    if any(k in t for k in ["lead", "head of", "manager", "director", "vp ", "architect"]):
        return "Lead"
    return ""


def read(filename):
    return pd.read_csv(JOBS_DIR / filename, dtype=str).fillna("")


def to_canonical(df: pd.DataFrame, **overrides) -> pd.DataFrame:
    """Return a DataFrame with exactly OUT_COLS, filling missing with ''."""
    n = len(df)
    data = {}
    for col in OUT_COLS:
        if col in overrides:
            val = overrides[col]
            data[col] = val.values if isinstance(val, pd.Series) else [str(val)] * n
        elif col in df.columns:
            data[col] = df[col].values
        else:
            data[col] = [""] * n
    return pd.DataFrame(data)

## Load and standardize each source

In [4]:
# ---------------------------------------------------------------------------
# 1. LinkedIn
# ---------------------------------------------------------------------------
li_raw = read("linkedin_jobs_standardized.csv")
print(f"LinkedIn     : {len(li_raw):4d} rows")

li = to_canonical(
    li_raw,
    source          = "linkedin",
    source_url      = li_raw["link"],
    job_title       = li_raw["title"],
    company_name    = li_raw["companyName"],
    location        = li_raw["location"],
    employment_type = li_raw["employmentType"],
    seniority_level = li_raw["seniorityLevel"],
    industries      = li_raw["industries"],
    posting_date    = li_raw["postedAt"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
    full_text       = li_raw["descriptionText"],
)

LinkedIn     :  992 rows


In [5]:
# ---------------------------------------------------------------------------
# 2. Staff.am
# ---------------------------------------------------------------------------
sa_raw = read("staffam_jobs_standardized.csv")
print(f"Staff.am     : {len(sa_raw):4d} rows")

sa = to_canonical(
    sa_raw,
    seniority_level = sa_raw["specialist_level"],
    industries      = "",
    posting_date    = sa_raw["posting_date"].apply(normalize_date),
    deadline        = sa_raw["deadline"].apply(normalize_date),
)

Staff.am     :   55 rows


In [6]:
# ---------------------------------------------------------------------------
# 3. job.am
# ---------------------------------------------------------------------------
ja_raw = read("jobam_jobs_standardized.csv")
print(f"job.am       : {len(ja_raw):4d} rows")

ja = to_canonical(
    ja_raw,
    seniority_level = ja_raw["experience_level"],
    industries      = "",
    posting_date    = "",
    deadline        = ja_raw["deadline"].apply(normalize_date),
    skills_tags     = "",
)

job.am       :   20 rows


In [7]:
# ---------------------------------------------------------------------------
# 4. Picsart
# ---------------------------------------------------------------------------
pi_raw = read("picsart_jobs_standardized.csv")
print(f"Picsart      : {len(pi_raw):4d} rows")

pi = to_canonical(
    pi_raw,
    employment_type = "",
    seniority_level = pi_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = pi_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

Picsart      :    2 rows


In [8]:
# ---------------------------------------------------------------------------
# 5. Krisp
# ---------------------------------------------------------------------------
kr_raw = read("krisp_jobs_standardized.csv")
print(f"Krisp        : {len(kr_raw):4d} rows")

kr = to_canonical(
    kr_raw,
    employment_type = kr_raw["work_type"],
    seniority_level = kr_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = kr_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

Krisp        :    7 rows


In [9]:
# ---------------------------------------------------------------------------
# 6. ServiceTitan
# ---------------------------------------------------------------------------
st_raw = read("servicetitan_jobs_standardized.csv")
print(f"ServiceTitan : {len(st_raw):4d} rows")

st = to_canonical(
    st_raw,
    employment_type = "",
    seniority_level = st_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = st_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

ServiceTitan :    4 rows


In [10]:
# ---------------------------------------------------------------------------
# 7. EPAM
# ---------------------------------------------------------------------------
ep_raw = read("epam_jobs_standardized.csv")
print(f"EPAM         : {len(ep_raw):4d} rows")

ep = to_canonical(
    ep_raw,
    employment_type = "",
    seniority_level = ep_raw["seniority_level"],
    industries      = ep_raw["department"],
    posting_date    = ep_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = ep_raw["skills_tags"],
)

EPAM         :  108 rows


In [11]:
# ---------------------------------------------------------------------------
# 8. SoftConstruct
# ---------------------------------------------------------------------------
sc_raw = read("softconstruct_jobs_standardized.csv")
print(f"SoftConstruct: {len(sc_raw):4d} rows")

sc = to_canonical(
    sc_raw,
    employment_type = "",
    seniority_level = sc_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = sc_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

SoftConstruct:  152 rows


In [12]:
# ---------------------------------------------------------------------------
# 9. DISQO
# ---------------------------------------------------------------------------
dq_raw = read("disqo_jobs_standardized.csv")
print(f"DISQO        : {len(dq_raw):4d} rows")

dq = to_canonical(
    dq_raw,
    employment_type = "",
    seniority_level = dq_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = dq_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

DISQO        :    1 rows


In [13]:
# ---------------------------------------------------------------------------
# 10. Synopsys
# ---------------------------------------------------------------------------
sy_raw = read("synopsys_jobs_standardized.csv")
print(f"Synopsys     : {len(sy_raw):4d} rows")

sy = to_canonical(
    sy_raw,
    posting_date = sy_raw["posting_date"].apply(normalize_date),
    deadline     = sy_raw["deadline"].apply(normalize_date),
)

Synopsys     :    2 rows


In [14]:
# ---------------------------------------------------------------------------
# 11. DataArt
# ---------------------------------------------------------------------------
da_raw = read("dataart_jobs_standardized.csv")
print(f"DataArt      : {len(da_raw):4d} rows")

da = to_canonical(
    da_raw,
    posting_date = da_raw["posting_date"].apply(normalize_date),
    deadline     = da_raw["deadline"].apply(normalize_date),
)

DataArt      :    5 rows


In [15]:
# ---------------------------------------------------------------------------
# 12. Grid Dynamics
# ---------------------------------------------------------------------------
gd_raw = read("griddynamics_jobs_standardized.csv")
print(f"GridDynamics : {len(gd_raw):4d} rows")

gd = to_canonical(
    gd_raw,
    employment_type = "",
    seniority_level = gd_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology / IT Services",
    posting_date    = "",
    deadline        = "",
    skills_tags     = "",
)

GridDynamics :   11 rows


In [16]:
# ---------------------------------------------------------------------------
# 13. NVIDIA
# ---------------------------------------------------------------------------
nv_raw = read("nvidia_jobs_standardized.csv")
print(f"NVIDIA       : {len(nv_raw):4d} rows")

nv = to_canonical(
    nv_raw,
    employment_type = "",
    seniority_level = nv_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology / Semiconductors",
    posting_date    = "",
    deadline        = "",
    skills_tags     = "",
)

NVIDIA       :    5 rows


In [17]:
# ---------------------------------------------------------------------------
# 14. 10Web
# ---------------------------------------------------------------------------
tw_raw = read("10web_jobs_standardized.csv")
print(f"10Web        : {len(tw_raw):4d} rows")

tw = to_canonical(
    tw_raw,
    employment_type = tw_raw["employment_type"],
    seniority_level = tw_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology / AI Website Builder",
    posting_date    = "",
    deadline        = "",
    skills_tags     = "",
)

10Web        :    5 rows


## Merge and save

In [18]:
merged = pd.concat(
    [li, sa, ja, pi, kr, st, ep, sc, dq, sy, da,
     gd, nv, tw],
    ignore_index=True
)

# source_type: aggregator vs company_portal
merged["source_type"] = merged["source"].apply(
    lambda s: "aggregator" if s in AGGREGATORS else "company_portal"
)

merged = merged[OUT_COLS].fillna("")

out_path = JOBS_DIR / "final_jobs_dataset.csv"
merged.to_csv(out_path, index=False, encoding="utf-8")

print(f"\nMerged saved : {out_path}")
print(f"Total rows   : {len(merged)}")


Merged saved : /Users/lianaaghamalyan/thesis_data/data/processed/jobs/final_jobs_dataset.csv
Total rows   : 1369


## Summary statistics

In [19]:
print()
print("=" * 55)
print(f"{'SOURCE':<16} {'ROWS':>6}  {'FULL_TEXT':>10}  {'SENIORITY':>10}")
print("=" * 55)
for src, grp in merged.groupby("source"):
    ft_filled  = grp["full_text"].replace("", pd.NA).notna().sum()
    sen_filled = grp["seniority_level"].replace("", pd.NA).notna().sum()
    print(f"  {src:<14} {len(grp):>6}  {ft_filled:>5}/{len(grp):<4}  {sen_filled:>5}/{len(grp)}")
print("=" * 55)
print(f"  {'TOTAL':<14} {len(merged):>6}")
print()

print("=== FIELD COVERAGE ===")
for col in OUT_COLS:
    filled = merged[col].replace("", pd.NA).notna().sum()
    pct    = filled / len(merged) * 100
    print(f"  {col:20s}: {filled:5d}/{len(merged)} ({pct:.1f}%)")
print()

ft = merged["full_text"].replace("", pd.NA).dropna().str.len()
print(f"full_text (non-blank) — n={len(ft)}  Min={ft.min():.0f}  Median={ft.median():.0f}  Max={ft.max():.0f}")


SOURCE             ROWS   FULL_TEXT   SENIORITY
  10web               5      5/5         4/5
  dataart             5      5/5         5/5
  disqo               1      1/1         0/1
  epam              108    108/108     108/108
  griddynamics       11     11/11       10/11
  job.am             20     20/20       20/20
  krisp               7      7/7         5/7
  linkedin          992    992/992     931/992
  nvidia              5      5/5         4/5
  picsart             2      2/2         0/2
  servicetitan        4      4/4         3/4
  softconstruct     152    152/152      62/152
  staff.am           55     55/55       55/55
  synopsys            2      2/2         2/2
  TOTAL            1369

=== FIELD COVERAGE ===
  source              :  1369/1369 (100.0%)
  source_type         :  1369/1369 (100.0%)
  source_url          :  1369/1369 (100.0%)
  job_title           :  1369/1369 (100.0%)
  company_name        :  1369/1369 (100.0%)
  location            :  1369/1369 (100.0%)
